### Library and model imports (execute just once)

In [ ]:
from utils import list_images, check_marker_metadata_match, read_image, extract_img_metadata
from pathlib import Path
import napari
from cellpose import models, core
from skimage.segmentation import clear_border
from skimage.measure import regionprops_table
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

#Check if notebook has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

#Load pre-trained Cellpose-SAM
model = models.CellposeModel(gpu=True)

### Define OCs to analyze and their channel position

In [ ]:
# Define the channels you want to analyze using the following structure:
# markers = [(channel_name, channel_nr),(..., ...)]
# Remember in Python one starts counting from 0, so your first channel will be 0
# i.e. markers = [("SD_DAPI", 0), ("SD_RFP", 1)]

# This list hold the markers for signal intensity analysis 
markers = [("SD_DAPI", 1), ("SD_GFP", 2), ("SD_RFP", 3), ("SD_AF647", 4)]

# Define brightfield channel position to use as input for CellposeSAM-mediated cell segmentation
brightfield_channel = 0

### Define the path containing your images

In [ ]:
# Copy the path to the folder containing your images in between the quotation marks
data_folder = r"X:\Shirin\RCDAnalysis_Alberto\SK0052"

# If the path is correct you should see a list of the first 10 images in your folder down below
images = list_images(data_folder, format="nd2")
images[:10]

# Check if the defined marker position match the metadata in the image files
metadata_match = check_marker_metadata_match(images, markers)
if not metadata_match:
    raise RuntimeError(
        "Marker/metadata mismatch detected in one or more files. "
        "Fix markers or input data before continuing."
    )

### Define a single image file for exploration
Just edit the number in between brackets [ ] under <code>img_filepath = images[0]</code> to select the file

In [ ]:

# Extract experiment_id from data folder Path object
experiment_id = Path(data_folder).name

# Open any of the  images to analyze by changing the index number in between brackets []
# index number refers to the position of the file inside the images folder
# i.e. to open the first image [0], second [1] - in Python one starts counting from zero
img_filepath = images[0]
img, filename = read_image(img_filepath)

### Predict cell labels using base CellposeSAM (4.0)

In [ ]:
# Extract image metadata from filename
descriptor_dict = extract_img_metadata (img_filepath, verbose = True)

# Predict cell labels using CellposeSAM from brightfield image
cell_labels, flows, styles = model.eval(img[brightfield_channel], niter=1000) # need to check the arguments

# Visualize results in Napari
viewer = napari.Viewer(ndisplay=2)
viewer.add_image(img)

# Remove cell entities touching the image border
cell_labels = clear_border(cell_labels)
viewer.add_labels(cell_labels, opacity=0.5)

### Extract morphology and intensity features

In [ ]:
# Morphology: computed once (same label_image for all markers)
morphology_properties = [
    "label",
    "area",                          # number of voxels (volume in voxel units)
    "area_bbox",                     # volume of axis-aligned bounding box
    "area_convex",                   # volume of convex hull of the region
    "area_filled",                   # volume after filling holes
    "axis_major_length",             # length of major axis from inertia tensor (elongation)
    "axis_minor_length",             # length of minor axis (second principal axis in 3D)
    "equivalent_diameter_area",      # diameter of sphere with same volume as region
    "euler_number",                  # topology: objects + holes − tunnels (connectivity)
    "extent",                        # volume / bounding-box volume (fill of the box)
    "feret_diameter_max",            # maximum Feret (caliper) diameter
    "solidity",                      # volume / convex-hull volume (compact vs lobed)
    "inertia_tensor_eigvals",        # eigenvalues of inertia tensor (3 values: shape/orientation)
]

# Intensity: computed per marker (different intensity_image each time)
intensity_properties = [
    "label",
    "intensity_mean",
    "intensity_min",
    "intensity_max",
    "intensity_std",
]

In [ ]:
# Compute morphology once (label_image is the same for all markers)
props_morphology = regionprops_table(
    label_image=cell_labels,
    properties=morphology_properties,
)
props_df = pd.DataFrame(props_morphology)

# Loop through markers and extract intensity features only
for marker_name, ch_nr in markers:
    print(f"Analyzing channel: {marker_name} ...")

    props = regionprops_table(
        label_image=cell_labels,
        intensity_image=img[ch_nr],
        properties=intensity_properties,
    )

    intensity_df = pd.DataFrame(props)

    # Rename intensity columns with marker prefix
    prefix = f"{marker_name}"
    rename_map = {"label": "label"}
    for prop in intensity_properties:
        if prop == "label":
            continue
        if prop.startswith("intensity_"):
            suffix = prop.replace("intensity_", "")
            rename_map[prop] = f"{prefix}_{suffix}_int"
    intensity_df.rename(columns=rename_map, inplace=True)

    # Derived columns
    mean_col = rename_map["intensity_mean"]
    max_col = rename_map["intensity_max"]
    # Max / mean ratio (puncta vs diffuse signal)
    intensity_df[f"{prefix}_max_mean_ratio"] = intensity_df[max_col] / intensity_df[mean_col].replace(0, np.nan)

    # Merge intensity data into main dataframe
    props_df = props_df.merge(intensity_df, on="label")
    # Total marker content per cell = mean * area (area from morphology)
    props_df[f"{prefix}_sum_int"] = props_df[mean_col] * props_df["area"]

# Add each key-value pair from descriptor_dict to props_df at the specified position
insertion_position = 0
for key, value in descriptor_dict.items():
    props_df.insert(insertion_position, key, value)
    insertion_position += 1  # Increment position to maintain the order of keys in descriptor_dict


In [ ]:
props_df

### Sanity check of generated label areas and threshold selection based on morphology (i.e. size)
This can be applied after extracting all features

In [ ]:

props_df["area"].dropna().plot.hist(bins=100)
plt.show()

# Check individual area values to set a logical threshold
row = props_df[props_df["label"] == 219]
print("solidity:", row["solidity"].values[0], "area:", row["area"].values[0])

In [ ]:
# Filter out labels with a cell area smaller than X
# These are cells with incorrect segmentation 
props_df = props_df[props_df["area"] >= 2500]

props_df["area"].dropna().plot.hist(bins=100)
plt.show()

### Visual inspection of cell labels after filtering

In [ ]:
# Collect valid labels
valid_labels = props_df["label"].values

# Mask cell_labels: keep only valid labels (according to area)
cell_labels_filtered = cell_labels.copy()
cell_labels_filtered[~np.isin(cell_labels_filtered, valid_labels)] = 0

viewer.add_labels(cell_labels_filtered)